In [3]:
import pandas as pd
df=pd.read_csv("dataset.csv")

In [4]:
df = df[
    [
        "user_id",
        "product_id",
        "rating",
        "date",
        "review_text",
        "label"
    ]
]

In [5]:
df.head()

,user_id,product_id,rating,date,review_text,label
0,--1-HgcfL1vWbG-1hgn1oA,43rd1LKcZRIunySzbMsyLQ,4,2012-08-14,"GrEAT!!! foOD great service even though, PriCe...",1
1,--AGQM8GPHljgKdnLisXgQ,JDNZxz0ud7zhuPo5pqznMA,5,2008-08-22,"LOVE PLACE, FRIENDLY CLEAN FOOD DELICIOUS U CH...",1
2,--cDU5woxccqoHW5jzxJBw,sR4EOfPuI-at41uIxIZPhw,5,2009-07-17,I CAME BIRTHDAY DINNER?! LAST MONTH GREAT TIME...,1
3,--eQVss9nAx54FWsZHZgpA,boE4Ahsssqic7o5wQLI04w,5,2011-07-08,This place amazing love atmosphere menu style ...,0
4,--wLozIyZ3VXKsJAsHPiAQ,KomhK0JD5cleEW55YTw7MQ,5,2009-01-23,"Probably best, caNToNese!? CHICAGO, eSpECiaLlY...",1


In [6]:
df["date"] = pd.to_datetime(df["date"])

In [7]:
df["label"].value_counts()

label
0    4993
1    4933
Name: count, dtype: int64

In [8]:
df = df.dropna(subset=["review_text"])

df["review_text"] = df["review_text"].astype(str)

df["review_text"] = df["review_text"].str.strip()

df = df[df["review_text"] != ""]

In [9]:
df["review_length"] = df["review_text"].str.split().apply(len)

In [10]:
df["exclamation_count"] = df["review_text"].str.count("!")
df["question_count"] = df["review_text"].str.count(r"\?")

In [11]:
df["capital_ratio"] = (
    df["review_text"].str.count(r"[A-Z]") /
    df["review_text"].str.len()
)

In [12]:
review_freq = df.groupby("user_id").size()

df["review_frequency"] = df["user_id"].map(review_freq)

In [13]:
product_avg = df.groupby("product_id")["rating"].mean()

df["rating_deviation"] = df["rating"] - df["product_id"].map(product_avg)

In [14]:
df = df.sort_values(["user_id","date"])

df["time_gap"] = df.groupby("user_id")["date"].diff().dt.days

In [15]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [16]:
def extract_aspects(text):
    doc = nlp(text)

    aspects = []

    for chunk in doc.noun_chunks:
        aspects.append(chunk.lemma_.lower())

    return aspects

In [17]:
df["aspects"] = df["review_text"].apply(extract_aspects)

In [18]:
df["aspect_count"] = df["aspects"].apply(len)

df["aspect_density"] = df["aspect_count"] / df["review_length"]

In [19]:
import pandas as pd
import spacy
import nltk
from nltk.corpus import stopwords
from collections import Counter

nltk.download('stopwords')

nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\KIIT0001\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [20]:
df["review_text"] = df["review_text"].astype(str)

df["review_text"] = df["review_text"].str.lower()

df["review_text"] = df["review_text"].str.replace(r"http\S+", "", regex=True)

df["review_text"] = df["review_text"].str.replace(r"[^a-zA-Z\s]", "", regex=True)

df["review_text"] = df["review_text"].str.strip()

In [21]:
df["word_count"] = df["review_text"].str.split().apply(len)

In [22]:
def extract_aspects(text):
    
    doc = nlp(text)
    
    aspects = []
    
    for chunk in doc.noun_chunks:
        
        phrase = chunk.lemma_.strip().lower()
        
        if phrase in stop_words:
            continue
            
        if len(phrase) < 3:
            continue
            
        aspects.append(phrase)
    
    
    for token in doc:
        
        if token.pos_ == "NOUN":
            
            word = token.lemma_.lower()
            
            if word not in stop_words and len(word) > 2:
                aspects.append(word)
    
    
    return list(set(aspects))

In [23]:
df["aspects"] = df["review_text"].apply(extract_aspects)

In [24]:
df["aspect_count"] = df["aspects"].apply(len)

In [25]:
df["unique_aspect_count"] = df["aspects"].apply(lambda x: len(set(x)))

In [26]:
df["aspect_density"] = df["aspect_count"] / df["word_count"]

In [27]:
user_aspects = df.groupby("user_id")["aspects"].sum()

user_aspects = user_aspects.apply(lambda x: len(set(x)))

df["aspect_diversity_user"] = df["user_id"].map(user_aspects)

In [28]:
aspect_counter = Counter()

for aspects in df["aspects"]:
    aspect_counter.update(aspects)

top_aspects = [a for a,_ in aspect_counter.most_common(10)]

In [29]:
for aspect in top_aspects:
    
    df[f"aspect_{aspect}"] = df["aspects"].apply(
        lambda x: 1 if aspect in x else 0
    )

In [30]:
df["exclamation_count"] = df["review_text"].str.count("!")

df["question_count"] = df["review_text"].str.count(r"\?")

df["capital_ratio"] = (
    df["review_text"].str.count(r"[A-Z]") /
    df["review_text"].str.len()
)

In [31]:
df.to_csv("aspect_features_dataset.csv", index=False)